In [42]:
import pandas as pd
import os
from pathlib import Path

In [43]:
tracks_path = "../data/raw/fma_metadata/tracks.csv"
tracks = pd.read_csv(tracks_path, header=[0, 1], index_col=0)

tracks.head()

album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

In [44]:
tracks_subset = tracks[
    [
        ('track', 'title'),
        ('track', 'genre_top'),
    ]
].copy()

tracks_subset.head()

track          
                    title genre_top
track_id                           
2                    Food   Hip-Hop
3            Electric Ave   Hip-Hop
5              This World   Hip-Hop
10                Freeway       Pop
20        Spiritual Level       NaN

In [45]:
tracks_subset = tracks_subset.reset_index()
tracks_subset.columns = ['track_id', 'title', 'genre_top']

tracks_subset.head()

,track_id,title,genre_top
0,2,Food,Hip-Hop
1,3,Electric Ave,Hip-Hop
2,5,This World,Hip-Hop
3,10,Freeway,Pop
4,20,Spiritual Level,NaN


In [46]:
audio_root = Path("../data/raw/fma_small")

def make_mp3_path(track_id):
    track_str = f"{int(track_id):06d}"
    folder = track_str[:3]
    return str(audio_root / folder / f"{track_str}.mp3")

tracks_subset.head()

,track_id,title,genre_top
0,2,Food,Hip-Hop
1,3,Electric Ave,Hip-Hop
2,5,This World,Hip-Hop
3,10,Freeway,Pop
4,20,Spiritual Level,NaN


In [47]:
tracks_subset['mp3_path'] = tracks_subset['track_id'].apply(make_mp3_path)
tracks_subset['file_exists'] = tracks_subset['mp3_path'].apply(os.path.exists)

print("Before:", len(tracks_subset))
# print("After:", len(tracks_clean))

print(tracks_subset['mp3_path'].head())
print(tracks_subset['file_exists'].sum())

Before: 106574
0    ../data/raw/fma_small/000/000002.mp3
1    ../data/raw/fma_small/000/000003.mp3
2    ../data/raw/fma_small/000/000005.mp3
3    ../data/raw/fma_small/000/000010.mp3
4    ../data/raw/fma_small/000/000020.mp3
Name: mp3_path, dtype: object
8000


In [48]:
tracks_clean = tracks_subset[tracks_subset['file_exists']].copy()
tracks_clean = tracks_clean.dropna(subset=['genre_top'])

In [49]:
final_df = tracks_clean[['track_id', 'title', 'genre_top', 'mp3_path']].copy()

final_df.head()

,track_id,title,genre_top,mp3_path
0,2,Food,Hip-Hop,../data/raw/fma_small/000/000002.mp3
2,5,This World,Hip-Hop,../data/raw/fma_small/000/000005.mp3
3,10,Freeway,Pop,../data/raw/fma_small/000/000010.mp3
15,140,Queen Of The Wires,Folk,../data/raw/fma_small/000/000140.mp3
16,141,Ohio,Folk,../data/raw/fma_small/000/000141.mp3


In [50]:
output_path = "../data/cleaned/fma_cleaned_dataset.csv"
final_df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: ../data/cleaned/fma_cleaned_dataset.csv
